#지능형 고객 응대 chatbot 구현

In [2]:
#환경 설정
#필요한 라이브러리 임포트
import os
#환경 설정
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings,ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

C:\Users\Jinho\AppData\Local\Temp\ipykernel_23848\62944404.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
#KEY 설정 & #파일 불러오기
env_file_path = r"C:\Users\Jinho\work\.env"
###############################################
file_path = r"C:\Users\Jinho\work\manual.txt"
output = load_dotenv(env_file_path)
loader = TextLoader(file_path, encoding="utf-8")
doc = loader.load()
print(doc)

[Document(metadata={'source': 'C:\\Users\\Jinho\\work\\manual.txt'}, page_content="질문: 코드크래프터스 아카데미에 어떻게 가입하나요?\n답변: 저희 웹사이트 우측 상단의 '회원가입' 버튼을 클릭하시고, 이메일 주소와 비밀번호만 입력하면 간단하게 가입이 완료됩니다. 이메일 인증 후 바로 모든 서비스를 이용하실 수 있습니다.\n\n질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?\n답변: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.\n\n질문: 강의를 다 들으면 수료증을 받을 수 있나요?\n답변: 물론입니다. 각 강의의 모든 챕터를 100% 수강 완료하시면, 즉시 온라인으로 발급되는 공식 수료증을 받으실 수 있습니다. 수료증은 PDF 파일로 다운로드하여 소장하거나 제출용으로 활용하실 수 있습니다.\n\n질문: 어떤 결제 수단을 사용할 수 있나요?\n답변: 저희는 학습자님의 편의를 위해 신용카드, 실시간 계좌이체, 그리고 가상계좌 입금 등 다양한 결제 방법을 지원하고 있습니다. 원하시는 결제 수단을 선택하여 진행해 주세요.\n\n질문: 강의 동영상이 자꾸 끊기거나 재생되지 않아요.\n답변: 동영상 재생에 불편을 드려 죄송합니다. 우선 사용하고 계신 인터넷 브라우저의 캐시를 삭제해 보시거나, 다른 브라우저(크롬, 엣지 등)에서 재생해 보시길 권장합니다. 문제가 지속될 경우, 즉시 기술 지원팀에 문의해 주세요.\n\n질문: 강의 자료나 소스 코드는 어디서 다운로드받나요?\n답변: 각 강의의 첫 번째 챕터인 '강의 소개' 섹션과 각 챕터의 하단에서 관련 PDF 교안 및 전체 소스 코드를 압축 파일 형태로 다운로드하실 수 있습니다.\n\n질문: 수강 기간에 제한이 있나요?\n답변: 아니요, 저희 코드크래프터스 아카데미의 모든 강의는 한 번 구매하시면 평생 

In [4]:
#텍스트 분할


In [5]:
text_split = RecursiveCharacterTextSplitter(
    #나중에 분할에 기준을 설정할수있다.
   #separators=["\n\n","\n"," ",""],
    chunk_size=110,
    chunk_overlap=0
)

docs = text_split.split_documents(documents=doc)

print("분할된 문서:\n", docs)
print("="*50)
print("분할된 문서의 수:",len(docs))
print(f"분할된 첫 번째 문서:\n{docs[0]}")

분할된 문서:
 [Document(metadata={'source': 'C:\\Users\\Jinho\\work\\manual.txt'}, page_content='질문: 코드크래프터스 아카데미에 어떻게 가입하나요?'), Document(metadata={'source': 'C:\\Users\\Jinho\\work\\manual.txt'}, page_content="답변: 저희 웹사이트 우측 상단의 '회원가입' 버튼을 클릭하시고, 이메일 주소와 비밀번호만 입력하면 간단하게 가입이 완료됩니다. 이메일 인증 후 바로 모든 서비스를 이용하실 수 있습니다."), Document(metadata={'source': 'C:\\Users\\Jinho\\work\\manual.txt'}, page_content='질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?'), Document(metadata={'source': 'C:\\Users\\Jinho\\work\\manual.txt'}, page_content='답변: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.'), Document(metadata={'source': 'C:\\Users\\Jinho\\work\\manual.txt'}, page_content='질문: 강의를 다 들으면 수료증을 받을 수 있나요?'), Document(metadata={'source': 'C:\\Users\\Jinho\\work\\manual.txt'}, page_content='답변: 물론입니다. 각 강의의 모든 챕터를 100% 수강 완료하시면, 즉시 온라인으로 발급되는 공식 수료증을 받으실 수 있습니다. 수료증은 PDF 파일로 다운로드하여 소장하거나 제출용으로 활용하실'), Document(metadata={'source': 'C:\\Users\\Jinho\\work\\manual.txt

In [6]:
#모델 이름
model_name = "text-embedding-3-small"

#OpenAI 임베딩 모델 생성
model = OpenAIEmbeddings(
    model = model_name
)
"""
1. 데이터 저장 테이블 
   1) embedding_metadata: 원본 텍스트, 메타 데이터 
   2) embeddings_queue: 임베딩 벡터 
2. from_documents() 함수
   1) ChromaDB 전용 함수
   2) 입력: docs (분할된 문서 조각, Document의 리스트)와 임베딩 모델
   3) 기능: 각 Document의 텍스트를 임베딩하고 메모리에 저장 -> 결과물: 벡터 저장소(vectorstore)
3. as_retriever() 함수
   1) ChromaDB 전용 함수. 검색 도구 생성
   2) invoke 함수를 통해서 사용자 질문 입력
      - 사용자 질문을 받을 경우, 벡터 저장소에 저장된 텍스트 중에서 사용자의 질문과 가장 유사한 텍스트를 찾아주는 역할 수행
      - search_kwargs={'k': n} -> 사용자 질문과 가장 유사도가 높은 n개의 텍스트 데이터를 가져오도록 설정
"""
chroma_path = "chroma_db"
########################################
output1 = os.path.exists(chroma_path)
print("경로1 :",output1)
output2 = os.listdir(chroma_path)
print("경로2 :",output2)
########################################
if os.path.exists(chroma_path) and os.listdir(chroma_path):
   vectorstore = Chroma(
      persist_directory=chroma_path,
      embedding_function=model
   )
else:
   #미리 생성된, 임베딩 저장소 생성
   vectorstore = Chroma.from_documents(
      documents=docs,
      embedding=model,
      persist_directory=chroma_path
   )


경로1 : True
경로2 : ['c23a45ad-c0bf-4f75-b00e-ac11c7642783', 'chroma.sqlite3']


C:\Users\Jinho\AppData\Local\Temp\ipykernel_23848\2525081439.py:30: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [7]:
#검색 도구 생성
retriever = vectorstore.as_retriever(search_kwargs={'k':1})

user_query = "환불 해줘"
output = retriever.invoke(user_query)
print("검색 결과:",output)
"""
# 저장된 문서 ID 목록
ids = list(vectorstore.index_to_docstore_id.values())

# ID로 문서 조회
stored_docs = vectorstore.get_by_ids(ids)

print("저장된 문서 수:", len(stored_docs))

for document in stored_docs:
    print("문서:", document.page_content)
    print("메타데이터:", document.metadata)
"""

검색 결과: [Document(metadata={'source': 'C:\\Users\\Jinho\\work\\manual.txt'}, page_content='답변: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.')]


'\n# 저장된 문서 ID 목록\nids = list(vectorstore.index_to_docstore_id.values())\n\n# ID로 문서 조회\nstored_docs = vectorstore.get_by_ids(ids)\n\nprint("저장된 문서 수:", len(stored_docs))\n\nfor document in stored_docs:\n    print("문서:", document.page_content)\n    print("메타데이터:", document.metadata)\n'

In [13]:
#RAG 체인 생성


#시스템 프롬프트 생성
system_prompt = """
당신은 친절한 온라인 교육 플랫폼 직원입니다.
제공된 [컨텍스트]를 바탕으로 고객의 질문에 정중하게 답변해주세요.

[답변 규칙]
1. [컨텍스트]의 내용을 바탕으로 자연스럽게 답변을 재구성하거나 그대로 활용하세요.
2. 답변 시 '답변:'이라는 표현을 사용하지 마세요.
3. [컨텍스트]에 질문에 대한 정보가 없다면, '죄송하지만 문의하신 내용에 대해서는 아는 바가 없습니다.'라고 답변하세요.

[컨텍스트]
{context}
"""
#사용자 프롬프트 생성
user_prompt = "{question}"
#전체 프롬프트
prompt = ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("user",user_prompt)
])

print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\n당신은 친절한 온라인 교육 플랫폼 직원입니다.\n제공된 [컨텍스트]를 바탕으로 고객의 질문에 정중하게 답변해주세요.\n\n[답변 규칙]\n1. [컨텍스트]의 내용을 바탕으로 자연스럽게 답변을 재구성하거나 그대로 활용하세요.\n2. 답변 시 '답변:'이라는 표현을 사용하지 마세요.\n3. [컨텍스트]에 질문에 대한 정보가 없다면, '죄송하지만 문의하신 내용에 대해서는 아는 바가 없습니다.'라고 답변하세요.\n\n[컨텍스트]\n{context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


In [9]:
##################################
#LLM 생성 
llm = ChatOpenAI(model = 'gpt-5-nano')
##################################
#outputparser 생성
outputparser = StrOutputParser()
##################################
# Chain 생성 
rag_chain = prompt | llm | outputparser
##################################

In [ ]:
# Chatbot 실행 
# 대화형 ,while 정의 실행


print("프로그램 실행합니다.!/종료 (q)")
print("="*50)
while True:
    
    try:
        user_query = input("Q&A:")
        retriever_docs = retriever.invoke(user_query)
        #print(f"사용자 질문과 가장 유사한 문서 리스트:",retriever_docs)

        if user_query.lower() == "q":
            print("프로그램 종료")
            break
        
        else:
            print("->",retriever_docs)
            print("--log--")
            #저장
            context_text = retriever_docs[0].page_content
            print(context_text)
            
        print("=="*50)
        response = rag_chain.invoke(
            {
                'context':context_text,
                'question':user_query
            }
        )
    except Exception as e:
        print("ERRROR!",e)
        break
    



프로그램 실행합니다.!/종료 (q)
-> [Document(metadata={'source': 'C:\\Users\\Jinho\\work\\manual.txt'}, page_content='질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?')]
--log--
질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?
ERRROR! "Input to ChatPromptTemplate is missing variables {'context'}.  Expected: ['context', 'question'] Received: ['conttext', 'question']\nNote: if you intended {context} to be part of the string and not a variable, please escape it with double curly braces like: '{{context}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "
